Esercizio: implementa un semplice modello Sequential in Keras prevedere il prezzo della case di california housing (integrato in Kera). Collega l'addestramento a TensoBoard impostando una cartella di log dinamica con timestamp. Assicurati di attivare il salvataggio degli istogrammi dei pesi ogni epoca. Suggerimento: usa la callback 'tf.keras.callbacks.TensorBoard' e imposta 'histogram_freq=1'. Dopo il training, avvia TensorBoard e individua il grafo del modello.

In [9]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- CONFIGURAZIONE AMBIENTE ---
# Disattiva i log di sistema non necessari di TensorFlow (info e warning)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
# Disattiva le ottimizzazioni oneDNN per evitare differenze di precisione numerica trascurabili
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

def main():
    # 1. PREPARAZIONE DATI
    print("Caricamento dati...")
    # Carichiamo il dataset MNIST (60.000 immagini di cifre scritte a mano)
    housing = fetch_california_housing()
    x, y = housing.data, housing.target

    print("Shape x:", x.shape)
    print("Shape y:", y.shape)
    
    # Suddivisione in Training, Validation e Test set
    x_train_full, x_test, y_train_full, y_test = train_test_split(
        x, y, test_size=0.2, random_state=42
    )

    x_train, x_val, y_train, y_val = train_test_split(
        x_train_full, y_train_full, test_size=0.2, random_state=42
    )

    # NORMALIZZAZIONE: Fondamentale per dati tabellari con scale diverse
    # (es. popolazione vs numero di stanze)
    scaler=StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_val = scaler.transform(x_val)
    x_test = scaler.transform(x_test)

    # 2. DEFINIZIONE DEL MODELLO (Architettura della Rete)
    # Creiamo un modello sequenziale (uno strato dopo l'altro)
    model = models.Sequential([
        # L'input ora ha 8 caratteristiche (features)
        Input(shape=(x_train.shape[1],), name="Input_Layer"),
        
        # Non serve Flaten() perchè i dati sono già in formato tabellare (1D)
        #layers.Flatten(),
        
        layers.Dense(64, activation='relu', name="Hidden_Layer_1"),
        layers.Dense(64, activation='relu', name="Hidden_Layer_2"),
        layers.Dropout(0.1, name="Regularization"),
        
        # OUTPUT: 1 solo neurone senza attivazione (o 'linear') 
        # perché dobbiamo predire un valore continuo (prezzo), non una classe
        layers.Dense(1, name="Output_Layer")
    ])

    # COMPILAZIONE: Definiamo come il modello deve imparare
    model.compile(
        optimizer='adam',                # Algoritmo di ottimizzazione (molto efficiente)
        loss='mean_squared_error',       # Funzione di errore per problemi di regressione
        metrics=['mean_absolute_error'] # Metrica per valutare la performance
    )

    # 3. CONFIGURAZIONE TENSORBOARD (Monitoraggio)
    # Creiamo una cartella specifica basata sul timestamp per ogni esecuzione
    #%load_ext tensorboard
    log_dir = os.path.join(os.getcwd(), "logs", "fit", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
    
    # Il callback TensorBoard scriverà i log durante l'allenamento
    tensorboard_callback = tf.keras.callbacks.TensorBoard(
        log_dir=log_dir, 
        histogram_freq=1, # Calcola la distribuzione dei pesi ad ogni epoca
        write_graph=True,  # Salva il grafico della struttura del modello
        write_images=True # Visualizza i pesi del modello come immagini in TensorBoard.
    )

    # 4. TRAINING (Allenamento)
    print(f"I log verranno salvati in: {log_dir}")
    model.fit(
        x_train, y_train, 
        epochs=30,                  # La regressione può richiedere più epoche per convergere
        validation_data=(x_test, y_test), # Valuta la precisione su dati mai visti dopo ogni epoca
        callbacks=[tensorboard_callback]  # Attiva TensorBoard
    )
    # Valutazione finale
    test_loss = model.evaluate(x_test, y_test)
    print(f"\nLoss finale sul Test Set: {test_loss}")


if __name__ == "__main__":
    main()
# per attivare la tensorboard: tensorboard --logdir="logs/fit"

Caricamento dati...
Shape x: (20640, 8)
Shape y: (20640,)
I log verranno salvati in: c:\Users\uberti\iCloudDrive\iCloudDrive\Barbara\EPICODE\PYTHON\MODULO 4\3_TensorFlow_PyTorch\logs\fit\20260514-181545
Epoch 1/30
413/413 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.8759 - mean_absolute_error: 0.6396 - val_loss: 0.4505 - val_mean_absolute_error: 0.4702
Epoch 2/30
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.4422 - mean_absolute_error: 0.4703 - val_loss: 0.3889 - val_mean_absolute_error: 0.4470
Epoch 3/30
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.4017 - mean_absolute_error: 0.4501 - val_loss: 0.3670 - val_mean_absolute_error: 0.4284
Epoch 4/30
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.3809 - mean_absolute_error: 0.4366 - val_loss: 0.3885 - val_mean_absolute_error: 0.4210
Epoch 5/30
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.3642 - mean_absolute_error: 0.4267 - val_loss: 0.3579 - val_mean_absolute_error: 0.4151
Epoch 6/30
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/